In [1]:
!pip -q install numpy scikit-learn sentence-transformers
!python --version
!pip freeze > requirements.txt

Python 3.13.18


In [2]:
%%writefile lab6_search.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or not all(rec["text"].strip() for rec in records):
            raise ValueError("Provide nonempty records")
        self.records = records
        self.backend = backend
        corpus = [rec["text"] for rec in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(corpus)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(MODEL_NAME, device="cpu")
            self.check_length(corpus)
            self.matrix = self.model.encode(corpus, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        limit = self.model.max_seq_length
        for item in texts:
            if len(self.model.tokenizer(item, truncation=False)["input_ids"]) > limit:
                raise ValueError("Input exceeds encoder token limit")

    def _scores(self, query):
        if self.backend == "tfidf":
            vec = self.vectorizer.transform([query])
            if vec.nnz == 0:
                return None
            return (vec @ self.matrix.T).toarray()[0]
        self.check_length([query])
        vec = self.model.encode([query], normalize_embeddings=True)[0]
        return self.matrix @ vec

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        scores = self._scores(query)
        if scores is None:
            return []
        wanted = filters or {}
        pool = [i for i, rec in enumerate(self.records)
                if all(rec.get(key) == val for key, val in wanted.items())]
        pool.sort(key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in pool[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for rec, score in hits:
        print(rec["id"], f"score={score:.3f}")
        print(rec["text"])

Writing lab6_search.py


In [3]:
%%writefile scenario1_handbook.py
from lab6_search import Retriever, show_hits

entries = [
    {"id": "assessment:late", "source": "handbook.txt",
     "text": "Late submission is normally penalized. Students with illness "
             "may request an extension with medical evidence before the deadline."},
    {"id": "assessment:format", "source": "handbook.txt",
     "text": "Submit the assignment as a PDF through the learning portal."},
]

finder = Retriever(entries)
show_hits(finder.search("illness medical evidence late submission", k=1))

print("\n--- Experiment: paraphrase ---")
paraphrase = "I was sick; can I hand it in tomorrow?"
print("TF-IDF:")
show_hits(finder.search(paraphrase, k=2))
try:
    semantic = Retriever(entries, backend="embedding")
    print("Embedding:")
    show_hits(semantic.search(paraphrase, k=2))
except Exception as err:
    print("Embedding backend unavailable:", err)

Writing scenario1_handbook.py


In [4]:
!python scenario1_handbook.py

assessment:late score=0.506
Late submission is normally penalized. Students with illness may request an extension with medical evidence before the deadline.

--- Experiment: paraphrase ---
TF-IDF:
No matching records.
modules.json: 100% 349/349 [00:00<00:00, 1.04MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 315kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 18.4MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 186kB/s]
config.json: 100% 612/612 [00:00<00:00, 1.95MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:01<00:00, 122MB/s, 6.67MB/s  ] 
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 76.3MB/s, 8.23MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 81.6MB/s, 8.86MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 6324.49it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.66MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 12.8MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 34.6M

In [5]:
%%writefile scenario2_support.py
from lab6_search import Retriever, show_hits

entries = [
    {"id": "returns:damage", "region": "PK",
     "text": "For a damaged parcel, contact support within 48 hours "
             "and attach photographs of the item and packaging."},
    {"id": "returns:change", "region": "PK",
     "text": "For a change of mind, return unused items within 14 days."},
    {"id": "returns:damage:other", "region": "OTHER",
     "text": "Damaged parcel reports in the other region have a 72 hour window."},
]

finder = Retriever(entries)
question = "damaged parcel photographs"
show_hits(finder.search(question, k=1, filters={"region": "PK"}))

print("\n--- Experiment: no region filter ---")
show_hits(finder.search(question, k=2))

print("\n--- Experiment: 'My parcel arrived broken' ---")
broken = "My parcel arrived broken"
print("TF-IDF:")
show_hits(finder.search(broken, k=3))
try:
    semantic = Retriever(entries, backend="embedding")
    print("Embedding:")
    show_hits(semantic.search(broken, k=3))
except Exception as err:
    print("Embedding backend unavailable:", err)

Writing scenario2_support.py


In [6]:
!python scenario2_support.py

returns:damage score=0.373
For a damaged parcel, contact support within 48 hours and attach photographs of the item and packaging.

--- Experiment: no region filter ---
returns:damage score=0.373
For a damaged parcel, contact support within 48 hours and attach photographs of the item and packaging.
returns:damage:other score=0.252
Damaged parcel reports in the other region have a 72 hour window.

--- Experiment: 'My parcel arrived broken' ---
TF-IDF:
returns:damage:other score=0.244
Damaged parcel reports in the other region have a 72 hour window.
returns:damage score=0.193
For a damaged parcel, contact support within 48 hours and attach photographs of the item and packaging.
returns:change score=0.000
For a change of mind, return unused items within 14 days.
Loading weights: 100% 103/103 [00:00<00:00, 6280.08it/s]
Embedding:
returns:damage score=0.703
For a damaged parcel, contact support within 48 hours and attach photographs of the item and packaging.
returns:damage:other score=0.50

In [7]:
%%writefile scenario3_docs.py
from lab6_search import Retriever, show_hits

entries = [
    {"id": "auth:v2", "version": "v2",
     "text": "Renew an expired API token with POST /v2/token/refresh. "
             "Send the refresh_token field; never print its value in logs."},
    {"id": "auth:v1", "version": "v1",
     "text": "Renew an expired API token with POST /v1/auth/renew."},
]

finder = Retriever(entries)
question = "renew expired API token"

for title, ver, count in [("", "v2", 1),
                          ("\n--- Experiment: version v1 ---", "v1", 1),
                          ("\n--- Experiment: version v3 (does not exist) ---", "v3", 1)]:
    if title:
        print(title)
    show_hits(finder.search(question, k=count, filters={"version": ver}))

Writing scenario3_docs.py


In [8]:
!python scenario3_docs.py

auth:v2 score=0.431
Renew an expired API token with POST /v2/token/refresh. Send the refresh_token field; never print its value in logs.

--- Experiment: version v1 ---
auth:v1 score=0.669
Renew an expired API token with POST /v1/auth/renew.

--- Experiment: version v3 (does not exist) ---
No matching records.


In [9]:
%%writefile scenario4_workplace.py
from lab6_search import Retriever, show_hits

entries = [
    {"id": "remote:staff", "audience": "employee",
     "text": "Employees may request remote work with manager approval."},
    {"id": "remote:contractor", "audience": "contractor",
     "text": "Contractors may work remotely only when the signed contract "
             "explicitly permits remote work."},
]

finder = Retriever(entries)
show_hits(finder.search("remote work", k=1, filters={"audience": "contractor"}))

print("\n--- Experiment: no audience filter ---")
show_hits(finder.search("remote work", k=2))

Writing scenario4_workplace.py


In [10]:
!python scenario4_workplace.py

remote:contractor score=0.435
Contractors may work remotely only when the signed contract explicitly permits remote work.

--- Experiment: no audience filter ---
remote:contractor score=0.435
Contractors may work remotely only when the signed contract explicitly permits remote work.
remote:staff score=0.394
Employees may request remote work with manager approval.


In [11]:
%%writefile scenario5_equipment.py
from lab6_search import Retriever, show_hits

entries = [
    {"id": "printer:blank", "device": "printer",
     "text": "For blank printer pages: 1. Check ink level. "
             "2. Run the printer cleaning utility. "
             "3. Print a test page. Follow the device manual."},
    {"id": "scanner:blank", "device": "scanner",
     "text": "For a blank scanner image, check document placement "
             "and select the correct scanning source."},
]

finder = Retriever(entries)
show_hits(finder.search("blank pages", k=1, filters={"device": "printer"}))

print("\n--- Experiment: no device filter ---")
for phrase in ("blank pages", "blank image"):
    print("Query:", phrase)
    show_hits(finder.search(phrase, k=1))

Writing scenario5_equipment.py


In [12]:
!python scenario5_equipment.py

printer:blank score=0.278
For blank printer pages: 1. Check ink level. 2. Run the printer cleaning utility. 3. Print a test page. Follow the device manual.

--- Experiment: no device filter ---
Query: blank pages
printer:blank score=0.278
For blank printer pages: 1. Check ink level. 2. Run the printer cleaning utility. 3. Print a test page. Follow the device manual.
Query: blank image
scanner:blank score=0.370
For a blank scanner image, check document placement and select the correct scanning source.


In [13]:
%%writefile guided_tokens.py
import re


def split_both(text):
    return text.split(), re.findall(r"\b\w+\b", text)


sample = "I can't submit CS4944 late!"
spaced, wordlike = split_both(sample)
print("Whitespace:", spaced, len(spaced))
print("Word-like:", wordlike, len(wordlike))
print("Original:", sample)

print("\n--- Experiment ---")
for item in ["help@example.com", "", "یہ ایک مثال ہے"]:
    spaced, wordlike = split_both(item)
    print(repr(item), "| whitespace:", spaced, len(spaced),
          "| word-like:", wordlike, len(wordlike))

Writing guided_tokens.py


In [14]:
!python guided_tokens.py

Whitespace: ['I', "can't", 'submit', 'CS4944', 'late!'] 5
Word-like: ['I', 'can', 't', 'submit', 'CS4944', 'late'] 6
Original: I can't submit CS4944 late!

--- Experiment ---
'help@example.com' | whitespace: ['help@example.com'] 1 | word-like: ['help', 'example', 'com'] 3
'' | whitespace: [] 0 | word-like: [] 0
'یہ ایک مثال ہے' | whitespace: ['یہ', 'ایک', 'مثال', 'ہے'] 4 | word-like: ['یہ', 'ایک', 'مثال', 'ہے'] 4


In [15]:
%%writefile task1_tokens.py
import re

inputs = [
    "AI, search!",
    "I can't log in.",
    "help@example.com",
    "CS4944 version 2.1",
    "",
    "یہ ایک مثال ہے",
    "I do NOT need a reset.",
]

pattern = re.compile(r"\b\w+\b", re.UNICODE)
table = []

print("=== Whitespace vs word-like regex ===")
for item in inputs:
    by_space = item.split()
    by_regex = pattern.findall(item)
    table.append((item, len(by_space), len(by_regex)))
    print("Original   :", repr(item))
    print("Whitespace :", by_space, len(by_space))
    print("Word-like  :", by_regex, len(by_regex))
    print()

assert pattern.findall("") == []

print("=== Summary table (original | #whitespace | #word-like) ===")
for item, n_space, n_regex in table:
    print(f"{item!r:32} {n_space:>3} {n_regex:>3}")

print("\n=== Normalization risk ===")
negated = "I do NOT need a reset."
dropped = {"i", "do", "not", "a"}
reduced = " ".join(w.lower() for w in pattern.findall(negated) if w.lower() not in dropped)
print("Original  :", negated)
print("Aggressive:", reduced, "<- negation 'not' removed, meaning flipped")
print("Identifier lowercased:", "CS4944".lower(),
      "<- case-sensitive identifiers can be damaged; keep the original")

print("\n=== Model sub-word tokens (all-MiniLM-L6-v2) ===")
try:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    for item in inputs[:2]:
        pieces = encoder.tokenizer.tokenize(item)
        print(repr(item), pieces, len(pieces))
except Exception as err:
    print("Model tokenizer unavailable (record this limitation):", err)

Writing task1_tokens.py


In [16]:
!python task1_tokens.py

=== Whitespace vs word-like regex ===
Original   : 'AI, search!'
Whitespace : ['AI,', 'search!'] 2
Word-like  : ['AI', 'search'] 2

Original   : "I can't log in."
Whitespace : ['I', "can't", 'log', 'in.'] 4
Word-like  : ['I', 'can', 't', 'log', 'in'] 5

Original   : 'help@example.com'
Whitespace : ['help@example.com'] 1
Word-like  : ['help', 'example', 'com'] 3

Original   : 'CS4944 version 2.1'
Whitespace : ['CS4944', 'version', '2.1'] 3
Word-like  : ['CS4944', 'version', '2', '1'] 4

Original   : ''
Whitespace : [] 0
Word-like  : [] 0

Original   : 'یہ ایک مثال ہے'
Whitespace : ['یہ', 'ایک', 'مثال', 'ہے'] 4
Word-like  : ['یہ', 'ایک', 'مثال', 'ہے'] 4

Original   : 'I do NOT need a reset.'
Whitespace : ['I', 'do', 'NOT', 'need', 'a', 'reset.'] 6
Word-like  : ['I', 'do', 'NOT', 'need', 'a', 'reset'] 6

=== Summary table (original | #whitespace | #word-like) ===
'AI, search!'                      2   2
"I can't log in."                  4   5
'help@example.com'                 1   3
'CS4

In [17]:
%%writefile guided_chatbot.py
import argparse

from lab6_search import Retriever

FAQ = [
    {"id": "faq:password", "question": "How do I reset my password?",
     "answer": "Use the recovery link on the login page.",
     "source": "it_help.txt"},
    {"id": "faq:submit", "question": "Where do I submit my assignment?",
     "answer": "Use the assignment upload page in the learning portal.",
     "source": "assessment.txt"},
    {"id": "faq:late", "question": "Can illness justify a late submission?",
     "answer": "Request an extension with medical evidence before the deadline.",
     "source": "handbook.txt"},
    {"id": "faq:linkvalid",
     "question": "How long does the recovery link stay valid?",
     "answer": "The recovery link stays valid for thirty minutes.",
     "source": "it_help.txt"},
    {"id": "faq:fileformat",
     "question": "Which file format should I use for my final file?",
     "answer": "Submit the final file as a PDF.",
     "source": "assessment.txt"},
]


def build_bot(backend="tfidf"):
    rows = [{**entry, "text": entry["question"] + " " + entry["answer"]} for entry in FAQ]
    return Retriever(rows, backend)


def reply(engine, query):
    cleaned = query.strip()
    if not cleaned:
        return "Please enter a question."
    for entry in FAQ:
        if cleaned.casefold() == entry["question"].casefold():
            return f"{entry['answer']} [source: {entry['source']}]"
    matches = engine.search(query)
    if not matches:
        return "No matching FAQ. Please contact the help desk."
    out = ["Possible matches; choose or rephrase your question:"]
    out += [f"{rec['question']} (similarity {score:.3f})" for rec, score in matches]
    return "\n".join(out)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["tfidf", "embedding"], default="tfidf")
    options = parser.parse_args()
    engine = build_bot(options.backend)
    print("FAQ assistant. Type exit to stop.")
    while True:
        try:
            typed = input("You: ")
        except (EOFError, KeyboardInterrupt):
            break
        if typed.strip().lower() == "exit":
            break
        print("Bot:", reply(engine, typed))


if __name__ == "__main__":
    main()

Writing guided_chatbot.py


In [18]:
from guided_chatbot import build_bot, reply
bot = build_bot()
for q in ['How do I reset my password?', 'How long does the recovery link stay valid?', 'password recovery', 'I was sick can I hand it in tomorrow', 'volcano geology', '']:
    print('You:', q)
    print('Bot:', reply(bot, q))
    print()

You: How do I reset my password?
Bot: Use the recovery link on the login page. [source: it_help.txt]

You: How long does the recovery link stay valid?
Bot: The recovery link stays valid for thirty minutes. [source: it_help.txt]

You: password recovery
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.427)
How long does the recovery link stay valid? (similarity 0.243)
Where do I submit my assignment? (similarity 0.000)

You: I was sick can I hand it in tomorrow
Bot: Possible matches; choose or rephrase your question:
Where do I submit my assignment? (similarity 0.198)
Can illness justify a late submission? (similarity 0.194)
How do I reset my password? (similarity 0.000)

You: volcano geology
Bot: No matching FAQ. Please contact the help desk.

You: 
Bot: Please enter a question.



In [19]:
%run guided_chatbot.py

FAQ assistant. Type exit to stop.
You: exit


In [20]:
%%writefile task1_chatbot_check.py
from guided_chatbot import build_bot, reply

bot = build_bot()
assert "it_help.txt" in reply(bot, "How do I reset my password?")
assert reply(bot, "") == "Please enter a question."
assert "No matching FAQ" in reply(bot, "volcano geology")
assert "Possible matches" in reply(bot, "password recovery")
print("FAQ behaviour checks passed")

Writing task1_chatbot_check.py


In [21]:
!python task1_chatbot_check.py

FAQ behaviour checks passed


In [22]:
from guided_chatbot import build_bot, reply
bot = build_bot('embedding')
for q in ['I forgot my login', 'late and password', 'volcano geology']:
    print('You:', q)
    print('Bot:', reply(bot, q))
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

You: I forgot my login
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.588)
How long does the recovery link stay valid? (similarity 0.172)
Where do I submit my assignment? (similarity 0.136)

You: late and password
Bot: Possible matches; choose or rephrase your question:
How do I reset my password? (similarity 0.424)
How long does the recovery link stay valid? (similarity 0.309)
Can illness justify a late submission? (similarity 0.294)

You: volcano geology
Bot: Possible matches; choose or rephrase your question:
Where do I submit my assignment? (similarity 0.071)
How do I reset my password? (similarity 0.063)
Which file format should I use for my final file? (similarity 0.027)



In [23]:
import shutil
from google.colab import files
shutil.make_archive('Task1', 'zip', '.')
files.download('Task1.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>